# 13 · Análisis exploratorio: banco de pruebas para la historia

**Etapa 16 del plan.** Cada idea se juzgaba en 201909 y 201910, con unos 6 M de error por mes. Este notebook evalúa seis meses, sobre clientes que el modelo no vio, y no vuelve a tocar esos dos meses.

**Preguntas:**
1. ¿El banco mide bien? Tiene que reproducir lo que ya se sabe: la historia vale mucho y las columnas de ruido no suman.
2. ¿Cuánto ruido tiene?
3. ¿Alguna de las cuatro ideas nuevas suma sobre `ACH`?

**Todo lo de este notebook es exploratorio.** No cambia el resultado de la regla pre-registrada.

**No carga 201909 ni 201910,** ni sus filas ni su clase. **No escribe en `exp/03/` a `exp/12/` ni en los `resultados/03_*` a `12_*`:** solo los lee.

Ficha: `fichas/etapa_16_banco_de_pruebas.md` (en el laboratorio).

## 1. Parámetros

**Retomar:** si la corrida se corta, se vuelve a correr entera. Los modelos ya hechos no se repiten.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import resource
import sys
import time
import warnings

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP_03 = Path('../exp/03')                          # solo se lee
EXP_06 = Path('../exp/06')                          # solo se lee
EXP = Path('../exp/13')
FIGURAS = RESULTADOS / 'figuras'

MES_TOPE = 201908                                   # nada posterior entra a este notebook
MESES_BANCO = [201903, 201904, 201905, 201906, 201907, 201908]
MESES_CON_FOTOS = [201905, 201906, 201907, 201908]  # promedio aparte: 201903 y 201904 tienen pocas fotos previas
MESES_TRAIN = [201903, 201904, 201905, 201906]      # solo para el control 5
MES_VALID = 201908
MESES_EVAL = [MES_VALID]

CONTROL = 'AC'
SIN_LAGS = 'ACS'
BASE_CURADA = 'ACH'
RUIDO = 'ACK'
NUEVAS = {'ACH': {'lags': [], 'tendencia': None, 'curada': True}}
CANDIDATAS = ['ACH_V', 'ACH_Q', 'ACH_T', 'ACH_Z']
CONFIGS = [SIN_LAGS, CONTROL, BASE_CURADA, RUIDO] + CANDIDATAS
CONTRASTES = {SIN_LAGS: [CONTROL], BASE_CURADA: [CONTROL, RUIDO] + CANDIDATAS}
HIPOTESIS = {
    'ACS': 'Calibración: sin lags ni deltas pierde claramente contra AC',
    'AC': 'Calibración: pierde por poco contra ACH, o empata',
    'ACH': 'Base de la comparación',
    'ACK': 'Calibración: columnas de ruido, cerca de 0 o negativo contra ACH',
    'ACH_V': 'Quien se va sin caer puede volverse errático antes: std3 del nivel y del delta',
    'ACH_Q': 'Un quiebre corto se pierde en promedios y pendientes: contraste de 4 meses, peor delta y cantidad de caídas',
    'ACH_T': 'Importa la distancia a su mejor momento, aunque sea de hace más de 4 meses: contra el máximo y la media de todas las fotos previas, y meses desde el máximo',
    'ACH_Z': 'Un delta en pesos no significa lo mismo con inflación alta: percentil del delta 1 y 2 dentro del mes, con el cero fijo',
}
EFECTO_BUSCADO = 5.0                                # M por mes
FOTOS = 7                                           # 201908 llega hasta 201901

N_GRUPOS = 5
N_CURADAS = 30
N_RUIDO = 90
MINIMO_PREVIOS = 2
BLOQUE = 20_000                                     # filas por bloque al armar las tablas

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 2
N_BOOTSTRAP = 2_000
BANDA = (3_000, 6_000)                              # ambos incluidos
EPSILON = 1e-12
POSITIVOS = ['BAJA+1', 'BAJA+2']
N_HILOS = 8

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

PRED = EXP / 'pred'
VARIABLES_CSV = RESULTADOS / '08_variables.csv'     # solo se lee
for d in (PRED, FIGURAS):
    d.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14

ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116, 201905: 126_763,
    201906: 128_370, 201907: 129_853, 201908: 131_775,
}
ESPERADO_BANCO = (768_499, 3_795)                   # filas, BAJA+2
ESPERADO_N_VARIABLES = {'ACS': 152, 'AC': 760, 'ACH': 880}
N_MES = 152
ESPERADO_ROTOS = 23


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


TODAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()[:10]
assert TODAS == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
SEMILLAS = TODAS[:N_SEMILLAS]


def huella_archivos():
    """Control 8: huella de las salidas anteriores, para comprobar que no cambian."""
    return {p.name: hashlib.blake2b(p.read_bytes(), digest_size=16).hexdigest()
            for p in sorted([*RESULTADOS.glob('0[3456789]_*'), *RESULTADOS.glob('1[012]_*')])}


def memoria_gb():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(r / (1e9 if sys.platform == 'darwin' else 1e6), 1)


HUELLA_INICIAL = huella_archivos()
T_INICIO = time.time()
print('Salidas anteriores registradas:', len(HUELLA_INICIAL))

## 2. Dataset y funciones de evaluación

**Control 1:** filas y `BAJA+2` del banco. **Control 2:** nada posterior a 201908 entra al notebook.

In [ ]:
def curva(prob, es_b2):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:LARGO_CURVA].astype(np.float64)


def banda(cv):
    return cv[..., BANDA[0] - 1:BANDA[1]].mean(axis=-1)


HIPER = json.loads((RESULTADOS / '03_hiperparametros.json').read_text())

t0 = time.time()
con = duckdb.connect()
base = con.execute(f"""
    select * from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {MES_TOPE}
    order by foto_mes, numero_de_cliente
""").df()
con.close()
assert base['foto_mes'].max() == MES_TOPE                                 # control 2
assert base['foto_mes'].value_counts().sort_index().to_dict() == ESPERADO_FILAS_POR_MES

CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]
BANDERAS = ['active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
            'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones']
ESTADOS = ['tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency']
FECHAS = [f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')]
EDADES = ['cliente_edad', 'cliente_antiguedad']
MONTOS = [v for v in VARIABLES if v not in BANDERAS + ESTADOS + FECHAS + EDADES]
assert len(VARIABLES) == 152 and len(MONTOS) == 129

rotos = pd.read_csv(RESULTADOS / '01_meses_rotos.csv')[['variable', 'foto_mes']]
assert len(rotos) == ESPERADO_ROTOS
for v, mes in rotos.itertuples(index=False):
    base.loc[base['foto_mes'] == mes, v] = np.nan

FOTO_MES = base['foto_mes'].to_numpy()
PERIODO = (FOTO_MES // 100 * 12 + FOTO_MES % 100).astype(np.int64)
CLAVE = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + PERIODO
INDICE = pd.Index(CLAVE)
assert INDICE.is_unique
X_MES = np.asfortranarray(base[VARIABLES].to_numpy(np.float32))
CLASE = base['clase_ternaria'].to_numpy()
del base
gc.collect()
print(f'Dataset cargado en {time.time() - t0:.0f} s')

## 3. Tablas

`ACH` se arma con la misma función del `12`. `AC` son sus primeras 760 columnas y `ACS` sus primeras 152.

**Columnas nuevas,** sobre las 30 variables curadas:

| Bloque | Columnas |
|---|---|
| `ACH_V`: inestabilidad | `std3` (desvío de 3 meses) y `dstd3` (desvío de los últimos 3 deltas) |
| `ACH_Q`: quiebre reciente | `q4` (media de los 2 últimos meses menos la de los 2 anteriores), `peor3` (peor delta de 3) y `caidas3` (cantidad de caídas en 3) |
| `ACH_T`: toda la historia | `relmaxT` y `relmediaT` (el mes contra el máximo y la media de todas las fotos previas) y `desdemaxT` (meses desde el máximo) |
| `ACH_Z`: contra el resto del mes | `zc1` y `zc2`: percentil del delta 1 y 2 dentro del mes, con el cero fijo |

`ACK` suma 90 columnas de `ACH`, permutadas dentro del mes, como en el `09`.

**Control 3:** cada cliente está en un solo grupo. **Control 4:** una columna da lo mismo calculada solo con datos hasta su mes. **Control 6:** fórmulas a mano en 300 clientes.

In [ ]:
SIN_HISTORIA = {'lags': [], 'tendencia': None, 'curada': False}


def receta(config):
    return SIN_HISTORIA if config == CONTROL else NUEVAS[config]


def especificacion(config):
    ex = receta(config)
    e = [('original', v, 0) for v in VARIABLES]
    e += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
    for k in ex['lags']:
        e += [(op, v, k) for op in ('lag', 'delta') for v in MONTOS + BANDERAS]
        e += [(op, v, k) for op in ('lag', 'cambio') for v in ESTADOS]
    if ex['tendencia']:
        h = ex['tendencia']
        e += [(op, v, h) for op in ('pend', 'prom', 'min', 'max') for v in MONTOS]
        e += [('perm', v, h) for v in BANDERAS]
    if ex['curada']:
        assert len(CURADAS) == N_CURADAS
        e += [(op, v, k) for op, k in (('ratio', 3), ('z', 4), ('relmax', 4), ('mes_ratio', 3)) for v in CURADAS]
    return e


def nombre(op, v, k):
    return v if op == 'original' else f'{v}_{op}{k}'


def construir(config, meses=None):
    ex = receta(config)
    h = ex['tendencia']
    h_max = max([2] + ex['lags'] + [h or 0] + ([4] if ex['curada'] else []))
    filas = np.flatnonzero(np.isin(FOTO_MES, meses or MESES_TRAIN + MESES_EVAL))
    mes_fila = FOTO_MES[filas]
    n_train = int(np.isin(mes_fila, MESES_TRAIN).sum())
    assert meses or (np.isin(mes_fila[:n_train], MESES_TRAIN).all() and np.isin(mes_fila[n_train:], MESES_EVAL).all())
    pos = {k: INDICE.get_indexer(CLAVE[filas] - k) for k in range(h_max + 1)}

    espec = especificacion(config)
    ubic = {e: i for i, e in enumerate(espec)}
    F = np.empty((len(filas), len(espec)), dtype=np.float32)
    t = -np.arange((h or 0) + 1, dtype=np.float64)[:, None]
    # Por bloques de filas: escribir columna por columna recorre toda la tabla en cada pasada,
    # y con una tabla de más de ~4,5 GB eso tardó decenas de minutos.
    with np.errstate(invalid='ignore', divide='ignore'):
        for inicio in range(0, len(filas), BLOQUE):
            fb = slice(inicio, inicio + BLOQUE)
            for j, v in enumerate(VARIABLES):
                F[fb, ubic[('original', v, 0)]] = X_MES[filas[fb], j]
                hk = {}
                for k in range(h_max + 1):
                    col = X_MES[pos[k][fb], j]
                    col[pos[k][fb] < 0] = np.nan
                    hk[k] = col
                for k in range(1, h_max + 1):
                    if ('lag', v, k) in ubic:
                        F[fb, ubic[('lag', v, k)]] = hk[k]
                    if ('delta', v, k) in ubic:
                        F[fb, ubic[('delta', v, k)]] = hk[0] - hk[k]
                    if ('cambio', v, k) in ubic:
                        a, b = hk[0], hk[k]
                        F[fb, ubic[('cambio', v, k)]] = np.where(np.isnan(a) | np.isnan(b), np.nan, (a != b).astype(np.float32))
                if h and (('prom', v, h) in ubic or ('perm', v, h) in ubic):
                    S = np.stack([hk[k] for k in range(h + 1)]).astype(np.float64)
                    valido = ~np.isnan(S)
                    cuantos = valido.sum(axis=0)
                    S0 = np.where(valido, S, 0.0)
                    if ('perm', v, h) in ubic:
                        F[fb, ubic[('perm', v, h)]] = np.where(cuantos > 0, S0.sum(axis=0), np.nan)
                    else:
                        prom = S0.sum(axis=0) / cuantos
                        dt = np.where(valido, t - (valido * t).sum(axis=0) / cuantos, 0.0)
                        pend = (dt * (S0 - prom)).sum(axis=0) / (dt ** 2).sum(axis=0)
                        pend[cuantos < 2] = np.nan
                        F[fb, ubic[('pend', v, h)]] = pend
                        F[fb, ubic[('prom', v, h)]] = prom
                        F[fb, ubic[('min', v, h)]] = np.where(cuantos > 0, np.where(valido, S, np.inf).min(axis=0), np.nan)
                        F[fb, ubic[('max', v, h)]] = np.where(cuantos > 0, np.where(valido, S, -np.inf).max(axis=0), np.nan)
                if ('ratio', v, 3) in ubic:
                    a, b, c = (hk[k].astype(np.float64) for k in range(3))
                    media3 = (a + b + c) / 3                  # NA si falta alguna foto
                    ratio = np.where(np.abs(media3) > EPSILON, a / media3, np.nan)
                    F[fb, ubic[('ratio', v, 3)]] = ratio
                    x = a
                    P = np.stack([hk[k] for k in range(1, 5)]).astype(np.float64)
                    valido = ~np.isnan(P)
                    cuantos = valido.sum(axis=0)
                    alcanza = (cuantos >= MINIMO_PREVIOS) & ~np.isnan(x)
                    media = np.where(valido, P, 0.0).sum(axis=0) / cuantos
                    desvio = np.sqrt(np.where(valido, (P - media) ** 2, 0.0).sum(axis=0) / cuantos)
                    maximo = np.where(valido, P, -np.inf).max(axis=0)
                    F[fb, ubic[('z', v, 4)]] = np.where(alcanza & (desvio > EPSILON), (x - media) / desvio, np.nan)
                    F[fb, ubic[('relmax', v, 4)]] = np.where(alcanza & (maximo > EPSILON), x / maximo, np.nan)
    for v in (CURADAS if ex['curada'] else []):           # el percentil necesita el mes entero
        F[:, ubic[('mes_ratio', v, 3)]] = pd.Series(F[:, ubic[('ratio', v, 3)]]).groupby(mes_fila).rank(pct=True).to_numpy(np.float32)
    return {'F': F, 'columnas': [nombre(*e) for e in espec], 'espec': espec, 'filas': filas, 'mes_fila': mes_fila,
            'n_train': n_train, 'lags': sorted({k for op, _, k in espec if op == 'lag'}), 'h': h}


CURADAS = pd.read_csv(VARIABLES_CSV)['variable'].tolist()
assert len(CURADAS) == N_CURADAS and set(CURADAS) <= set(MONTOS)


def cero_fijo(x, mes):
    """Percentil dentro del mes: positivos de 0 a 1, negativos de 0 a -1, y el cero en 0."""
    s = pd.Series(x)
    arriba = s.where(s > 0).groupby(mes).rank(pct=True).fillna(0)
    abajo = (-s).where(s < 0).groupby(mes).rank(pct=True).fillna(0)
    return (arriba - abajo).where(s.notna()).to_numpy()


def bloques_nuevos(filas, X, indice, clave, foto_mes):
    """Cada fila usa solo fotos de su mes o anteriores. El percentil usa solo su mes."""
    pos = [indice.get_indexer(clave[filas] - k) for k in range(FOTOS + 1)]
    mes = foto_mes[filas]
    salida = {b: {} for b in CANDIDATAS}
    with np.errstate(invalid='ignore', divide='ignore'):
        for v in CURADAS:
            j = VARIABLES.index(v)
            H = np.stack([np.where(p >= 0, X[p, j], np.nan) for p in pos]).astype(np.float64)
            a, b, c, d = H[:4]
            dif = np.stack([a - b, b - c, c - d])
            valido = ~np.isnan(dif)
            cuantos = valido.sum(axis=0)
            alcanza = cuantos >= MINIMO_PREVIOS
            media = np.where(valido, dif, 0.0).sum(axis=0) / cuantos
            desvio = np.sqrt(np.where(valido, (dif - media) ** 2, 0.0).sum(axis=0) / cuantos)
            salida['ACH_V'][f'{v}_std3'] = np.stack([a, b, c]).std(axis=0)
            salida['ACH_V'][f'{v}_dstd3'] = np.where(alcanza, desvio, np.nan)
            salida['ACH_Q'][f'{v}_q4'] = (a + b) / 2 - (c + d) / 2
            salida['ACH_Q'][f'{v}_peor3'] = np.where(alcanza, np.where(valido, dif, np.inf).min(axis=0), np.nan)
            salida['ACH_Q'][f'{v}_caidas3'] = np.where(alcanza, (dif < 0).sum(axis=0), np.nan)
            P = H[1:]
            valido = ~np.isnan(P)
            cuantos = valido.sum(axis=0)
            alcanza = (cuantos >= MINIMO_PREVIOS) & ~np.isnan(a)
            maximo = np.where(valido, P, -np.inf).max(axis=0)
            media = np.where(valido, P, 0.0).sum(axis=0) / cuantos
            salida['ACH_T'][f'{v}_relmaxT'] = np.where(alcanza & (maximo > EPSILON), a / maximo, np.nan)
            salida['ACH_T'][f'{v}_relmediaT'] = np.where(alcanza & (np.abs(media) > EPSILON), a / media, np.nan)
            salida['ACH_T'][f'{v}_desdemaxT'] = np.where(alcanza, np.where(np.isnan(H), -np.inf, H).argmax(axis=0), np.nan)
            salida['ACH_Z'][f'{v}_zc1'] = cero_fijo(a - b, mes)
            salida['ACH_Z'][f'{v}_zc2'] = cero_fijo(a - c, mes)
    return {b: (np.column_stack(list(c.values())).astype(np.float32), list(c)) for b, c in salida.items()}


def bloque_ruido(T):
    rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
    elegidas = np.sort(rng.choice(T['F'].shape[1], size=N_RUIDO, replace=False))
    copia = np.ascontiguousarray(T['F'][:, elegidas])
    for mes in np.unique(T['mes_fila']):
        en_mes = T['mes_fila'] == mes
        copia[en_mes] = rng.permuted(copia[en_mes], axis=0)
    return copia, [f'ruido_{i:03d}' for i in range(N_RUIDO)]


t0 = time.time()
T = construir(BASE_CURADA, meses=MESES_BANCO)
assert T['F'].shape == (ESPERADO_BANCO[0], ESPERADO_N_VARIABLES[BASE_CURADA]), T['F'].shape
assert T['columnas'][:N_MES] == VARIABLES and T['columnas'][ESPERADO_N_VARIABLES[CONTROL]].endswith('_ratio3')
assert (CLASE[T['filas']] == 'BAJA+2').sum() == ESPERADO_BANCO[1]         # control 1
B = bloques_nuevos(T['filas'], X_MES, INDICE, CLAVE, FOTO_MES)
B[RUIDO] = bloque_ruido(T)
assert {b: v[0].shape[1] for b, v in B.items()} == {'ACH_V': 60, 'ACH_Q': 90, 'ACH_T': 90, 'ACH_Z': 60, 'ACK': 90}

# Control 3: grupos de clientes, fijos para todas las configuraciones
CLIENTE_FILA = CLAVE[T['filas']] // 100_000
CLIENTES, POS_CLIENTE = np.unique(CLIENTE_FILA, return_inverse=True)
grupo_cliente = np.random.default_rng(SEMILLA_PRIMIGENIA).permutation(len(CLIENTES)) % N_GRUPOS
ruta_grupos = EXP / 'grupos.npy'
if ruta_grupos.exists():
    assert np.array_equal(np.load(ruta_grupos), grupo_cliente), 'Cambiaron los grupos'
else:
    np.save(ruta_grupos, grupo_cliente)
GRUPO_FILA = grupo_cliente[POS_CLIENTE]
assert pd.Series(GRUPO_FILA).groupby(CLIENTE_FILA).nunique().max() == 1
b2_grupo = pd.Series(CLASE[T['filas']] == 'BAJA+2').groupby(GRUPO_FILA).sum()
assert b2_grupo.min() > 0.85 * ESPERADO_BANCO[1] / N_GRUPOS, b2_grupo.to_dict()
print('BAJA+2 por grupo:', b2_grupo.to_dict())

# Control 4: sin fuga. Con los datos cortados en el mes, cada columna tiene que dar lo mismo.
for corte in (201905, 201906):
    hasta = FOTO_MES <= corte
    filas_c = np.flatnonzero(FOTO_MES[hasta] == corte)
    C = bloques_nuevos(filas_c, X_MES[hasta], pd.Index(CLAVE[hasta]), CLAVE[hasta], FOTO_MES[hasta])
    en_mes = T['mes_fila'] == corte
    for b in CANDIDATAS:
        assert np.array_equal(C[b][0], B[b][0][en_mes], equal_nan=True), (b, corte)
    del C

# Control 6: fórmulas a mano
rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
muestra = rng.choice(len(T['filas']), size=300, replace=False)
col = {b: {c: i for i, c in enumerate(B[b][1])} for b in CANDIDATAS}


def igual(bloque, nombre_columna, fila, esperado):
    valor = B[bloque][0][fila, col[bloque][nombre_columna]]
    if esperado is None:
        assert np.isnan(valor), (nombre_columna, fila, valor)
    else:
        np.testing.assert_allclose(valor, esperado, rtol=1e-3, atol=1e-3, err_msg=nombre_columna)


for v in CURADAS[:3]:
    serie = pd.Series(X_MES[:, VARIABLES.index(v)].astype(np.float64), index=INDICE)
    for i in muestra:
        clave = CLAVE[T['filas'][i]]
        y = [serie.get(clave - k, np.nan) for k in range(FOTOS + 1)]
        igual('ACH_V', f'{v}_std3', i, None if np.isnan(y[:3]).any() else float(np.std(y[:3])))
        igual('ACH_Q', f'{v}_q4', i, None if np.isnan(y[:4]).any() else (y[0] + y[1]) / 2 - (y[2] + y[3]) / 2)
        dif = [y[k] - y[k + 1] for k in range(3)]
        hay = [x for x in dif if not np.isnan(x)]
        igual('ACH_V', f'{v}_dstd3', i, float(np.std(hay)) if len(hay) >= 2 else None)
        igual('ACH_Q', f'{v}_peor3', i, min(hay) if len(hay) >= 2 else None)
        igual('ACH_Q', f'{v}_caidas3', i, sum(x < 0 for x in hay) if len(hay) >= 2 else None)
        previos = [x for x in y[1:] if not np.isnan(x)]
        alcanza = len(previos) >= 2 and not np.isnan(y[0])
        igual('ACH_T', f'{v}_relmaxT', i, y[0] / max(previos) if alcanza and max(previos) > EPSILON else None)
        igual('ACH_T', f'{v}_relmediaT', i, y[0] / np.mean(previos) if alcanza and abs(np.mean(previos)) > EPSILON else None)
        igual('ACH_T', f'{v}_desdemaxT', i, int(np.nanargmax(y)) if alcanza else None)
v = CURADAS[0]
en_mes = T['mes_fila'] == 201906
d = T['F'][en_mes, T['columnas'].index(f'{v}_delta1')].astype(np.float64)
z = B['ACH_Z'][0][en_mes, col['ACH_Z'][f'{v}_zc1']]
assert (np.isnan(d) == np.isnan(z)).all() and (np.sign(d[~np.isnan(d)]) == np.sign(z[~np.isnan(z)])).all()
hay = d > 0
np.testing.assert_allclose(z[hay], stats.rankdata(d[hay]) / hay.sum(), rtol=1e-5)
print(f'Tablas y controles 3, 4 y 6 OK en {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB')

## 4. Control 5: `AC` reproduce el `06`

Con la ventana 201903–06 y la primera semilla, `AC` tiene que dar las predicciones del `06` en 201908. Comprueba las filas, las columnas y los hiperparámetros.

In [ ]:
FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}
MEJORES = {**FIJOS, **HIPER['params']}
ARBOLES = HIPER['arboles']
k_ac = ESPERADO_N_VARIABLES[CONTROL]
ruta_control = EXP / 'control_5.txt'
if not ruta_control.exists():
    t0 = time.time()
    entrena = np.isin(T['mes_fila'], MESES_TRAIN)
    y = np.isin(CLASE[T['filas'][entrena]], POSITIVOS).astype(np.int8)
    dtrain = lgb.Dataset(T['F'][entrena, :k_ac], label=y, feature_name=T['columnas'][:k_ac],
                         params={'max_bin': 31, 'feature_pre_filter': False}, free_raw_data=False)
    modelo = lgb.train({**MEJORES, 'seed': SEMILLAS[0]}, dtrain, num_boost_round=ARBOLES)
    prob = modelo.predict(T['F'][T['mes_fila'] == MES_VALID, :k_ac], num_threads=N_HILOS).astype(np.float32)
    meses_06 = np.load(EXP_03 / 'claves_eval.npz')['foto_mes']
    esperado = np.load(EXP_06 / 'pred' / CONTROL / f'{SEMILLAS[0]}.npy')[meses_06 == MES_VALID]
    assert np.array_equal(prob, esperado), 'AC no reproduce el 06'
    ruta_control.write_text('ok\n')
    del dtrain, modelo, prob, esperado
    gc.collect()
    print(f'Control 5 OK en {time.time() - t0:.0f} s')
else:
    print('Control 5 ya hecho')

## 5. Modelos

Cada modelo entrena con 4 grupos de clientes y predice el quinto. Todas las configuraciones usan los mismos grupos y las mismas semillas.

**Retomar:** cada predicción se guarda al terminar. Los bins usan una semilla fija, para que un modelo dé lo mismo aunque la corrida se corte.

**Control 7:** cada cliente-mes tiene predicción de cada configuración y semilla.

In [ ]:
def ruta(config, semilla, grupo):
    return PRED / config / str(semilla) / f'{grupo}.npy'


def tabla_de(config, filas):
    k = ESPERADO_N_VARIABLES.get(config, ESPERADO_N_VARIABLES[BASE_CURADA])
    extra = config if config in B else None
    X = T['F'][filas, :k]
    if extra:
        X = np.hstack([X, B[extra][0][filas]])
    return X, T['columnas'][:k] + (B[extra][1] if extra else [])


pendientes = [(c, s, g) for c in CONFIGS for s in SEMILLAS for g in range(N_GRUPOS) if not ruta(c, s, g).exists()]
print('Modelos pendientes:', len(pendientes))
Y = np.isin(CLASE[T['filas']], POSITIVOS).astype(np.int8)
BINS = {'max_bin': 31, 'feature_pre_filter': False, 'data_random_seed': SEMILLA_PRIMIGENIA}
segundos = {}
for config in CONFIGS:
    for g in range(N_GRUPOS):
        falta = [s for s in SEMILLAS if not ruta(config, s, g).exists()]
        if not falta:
            continue
        entrena = GRUPO_FILA != g
        X, columnas = tabla_de(config, entrena)
        dtrain = lgb.Dataset(X, label=Y[entrena], feature_name=columnas, params=BINS, free_raw_data=False).construct()
        X_grupo = tabla_de(config, ~entrena)[0]
        for semilla in falta:
            t0 = time.time()
            modelo = lgb.train({**MEJORES, 'seed': semilla, 'data_random_seed': SEMILLA_PRIMIGENIA}, dtrain, num_boost_round=ARBOLES)
            ruta(config, semilla, g).parent.mkdir(parents=True, exist_ok=True)
            np.save(ruta(config, semilla, g), modelo.predict(X_grupo, num_threads=N_HILOS).astype(np.float32))
            segundos.setdefault(config, []).append(time.time() - t0)
        print(f'{config} · grupo {g}: {len(columnas)} variables, {np.mean(segundos[config]):.0f} s por modelo · memoria máxima {memoria_gb()} GB')
        del X, X_grupo, dtrain, modelo
        gc.collect()

PUNTAJE = {}
for config in CONFIGS:
    P = np.full((len(SEMILLAS), len(T['filas'])), np.nan)
    for i, s in enumerate(SEMILLAS):
        for g in range(N_GRUPOS):
            P[i, GRUPO_FILA == g] = np.load(ruta(config, s, g))
    assert not np.isnan(P).any(), config                  # control 7
    PUNTAJE[config] = P

## 6. Comparaciones

**Medida:** para cada mes se juntan las predicciones de los 5 grupos y se calcula la ganancia en la banda. Después se promedian los meses y las semillas.

**Intervalo:** bootstrap pareado de clientes. Cada cliente aporta lo que deja en todos sus meses, y se remuestrean clientes, no filas.

**Error del promedio mensual:** el desvío de ese bootstrap. Es la medida real del ruido del banco.

In [ ]:
ES_B2 = CLASE[T['filas']] == 'BAJA+2'
es_mes = {m: T['mes_fila'] == m for m in MESES_BANCO}
B_MES = {(c, m): np.array([banda(curva(p[es_mes[m]], ES_B2[es_mes[m]])) / 1e6 for p in PUNTAJE[c]]) for c in CONFIGS for m in MESES_BANCO}

filas = [{'config': c, 'semilla': s, 'foto_mes': m, 'ganancia_banda_M': round(B_MES[(c, m)][i], 4),
          'auc': round(roc_auc_score(ES_B2[es_mes[m]], PUNTAJE[c][i][es_mes[m]]), 6)}
         for c in CONFIGS for m in MESES_BANCO for i, s in enumerate(SEMILLAS)]
meses = pd.DataFrame(filas)
meses.to_csv(RESULTADOS / '13_meses.csv', index=False)

posicion = np.arange(1, BANDA[1] + 1)
PESO = (BANDA[1] - np.maximum(posicion, BANDA[0]) + 1) / (BANDA[1] - BANDA[0] + 1)


def por_cliente(config, lista_meses):
    """Lo que cada cliente aporta al promedio mensual de la banda."""
    v = np.zeros(len(CLIENTES))
    for m in lista_meses:
        idx = np.flatnonzero(es_mes[m])
        for p in PUNTAJE[config]:
            orden = np.argsort(-p[idx], kind='stable')[:BANDA[1]]
            aporte = PESO * np.where(ES_B2[idx[orden]], GANANCIA_ACIERTO, -COSTO_ESTIMULO)
            np.add.at(v, POS_CLIENTE[idx[orden]], aporte)
    return v / (len(lista_meses) * len(SEMILLAS))


def remuestrear(D, n_veces, semilla):
    rng = np.random.default_rng(semilla)
    n = D.shape[0]
    con_valor = np.flatnonzero(D.any(axis=1))
    salida = np.empty((n_veces, D.shape[1]))
    for b in range(n_veces):
        veces = np.bincount(rng.integers(0, n, size=n), minlength=n)
        salida[b] = veces[con_valor] @ D[con_valor]
    return salida


media_meses = lambda c, lista: float(np.mean([B_MES[(c, m)].mean() for m in lista]))
filas = []
for contra, lista in CONTRASTES.items():
    resumen = {}
    for clave_meses, lista_meses in (('6', MESES_BANCO), ('4', MESES_CON_FOTOS)):
        base_c = por_cliente(contra, lista_meses)
        assert abs(base_c.sum() / 1e6 - media_meses(contra, lista_meses)) < 1e-6
        D = np.column_stack([por_cliente(c, lista_meses) - base_c for c in lista])
        resumen[clave_meses] = (D.sum(axis=0) / 1e6, remuestrear(D, N_BOOTSTRAP, SEMILLA_PRIMIGENIA + len(contra) + int(clave_meses)) / 1e6)
    for j, c in enumerate(lista):
        d6, u6 = resumen['6'][0][j], resumen['6'][1][:, j]
        d4, u4 = resumen['4'][0][j], resumen['4'][1][:, j]
        fila = {'config': c, 'contra': contra, 'd_M': d6, 'ic95_inf_M': np.percentile(u6, 2.5), 'ic95_sup_M': np.percentile(u6, 97.5),
                'error_M': u6.std(), 'a_favor': f'{(u6 > 0).mean() * 100:.1f} %',
                'd4_M': d4, 'ic95_inf4_M': np.percentile(u4, 2.5), 'ic95_sup4_M': np.percentile(u4, 97.5)}
        fila.update({f'd_{m}': (B_MES[(c, m)] - B_MES[(contra, m)]).mean() for m in MESES_BANCO})
        filas.append(fila)
comparacion = pd.DataFrame(filas)
comparacion.round(4).to_csv(RESULTADOS / '13_comparacion.csv', index=False)
print('EXPLORATORIO')
print(comparacion.round(2).to_string(index=False))
print()
print('Ganancia en la banda por mes (M, media de semillas):')
print(pd.DataFrame({c: {m: B_MES[(c, m)].mean() for m in MESES_BANCO} for c in CONFIGS}).T.assign(media=lambda t: t.mean(axis=1)).round(2).to_string())

## 7. Figuras

Tamaño 1920 × 1080 (16:9). Todas llevan el rótulo de exploratorias.

In [ ]:
TINTA_2, GRILLA, AZUL = '#52514e', '#e1e0d9', '#2a78d6'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA_2, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'text.color': '#0b0b0b',
                     'axes.grid': True, 'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 13, 'axes.titlesize': 15})
TAMANO, DPI = (12.8, 7.2), 150


def coma(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


NOMBRES = {'AC': 'lag y delta 1 y 2', 'ACK': 'ruido (control)', 'ACH_V': 'inestabilidad', 'ACH_Q': 'quiebre reciente',
           'ACH_T': 'toda la historia', 'ACH_Z': 'contra el resto del mes'}
LISTA = CONTRASTES[BASE_CURADA]
t = comparacion[comparacion['contra'] == BASE_CURADA].set_index('config').loc[LISTA]
y = np.arange(len(LISTA))[::-1]
etiquetas = [NOMBRES[c] for c in LISTA]
formato = FuncFormatter(lambda v, _: coma(v, 0 if float(v).is_integer() else 1))

fig, ejes = plt.subplots(1, 2, figsize=TAMANO, sharex=True)
for ax, (d, inf, sup), titulo in zip(ejes, (('d_M', 'ic95_inf_M', 'ic95_sup_M'), ('d4_M', 'ic95_inf4_M', 'ic95_sup4_M')),
                                     ('201903 a 201908', '201905 a 201908')):
    ax.axvline(0, color=TINTA_2, lw=1.2)
    ax.hlines(y, t[inf], t[sup], color=AZUL, lw=2)
    ax.scatter(t[d], y, s=70, color=AZUL, zorder=3, edgecolor='#fcfcfb')
    ax.set_yticks(y, etiquetas if ax is ejes[0] else [])
    ax.set_ylim(-0.7, len(LISTA) - 0.3)
    ax.xaxis.set_major_formatter(formato)
    ax.set_title(titulo, loc='left')
    ax.set_xlabel('Diferencia contra ACH (M$ por mes), IC 95 %')
fig.suptitle('Exploratorio: banco de pruebas, contra la historia curada', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '13_1_banco.png', dpi=DPI)
plt.show()

fig, ejes = plt.subplots(2, 3, figsize=TAMANO, sharex=True, sharey=True)
for ax, m in zip(ejes.ravel(), MESES_BANCO):
    ax.axvline(0, color=TINTA_2, lw=1.2)
    ax.scatter(t[f'd_{m}'], y, s=60, color=AZUL, zorder=3, edgecolor='#fcfcfb')
    ax.set_yticks(y, etiquetas)
    ax.set_ylim(-0.7, len(LISTA) - 0.3)
    ax.xaxis.set_major_formatter(formato)
    ax.set_title(str(m), loc='left')
for ax in ejes[1]:
    ax.set_xlabel('Diferencia contra ACH (M$)')
fig.suptitle('Exploratorio: banco de pruebas, mes por mes', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '13_2_por_mes.png', dpi=DPI)
plt.show()

## 8. Lecturas fijadas antes de correr

Primero se lee el banco. Si no pasa, las candidatas no se interpretan.

**Control 8:** las salidas anteriores no cambiaron.

In [ ]:
assert huella_archivos() == HUELLA_INICIAL, 'Cambiaron las salidas anteriores'   # control 8
print(f'Control de salidas OK · {(time.time() - T_INICIO) / 60:.1f} min en total · memoria máxima {memoria_gb()} GB')
if segundos:
    print('Segundos por modelo:', {c: round(float(np.mean(v)), 1) for c, v in segundos.items()})
print()
print('EXPLORATORIO: no cambia el resultado de la regla pre-registrada.')
print()

fila = lambda c, contra: comparacion[(comparacion['config'] == c) & (comparacion['contra'] == contra)].iloc[0]
historia_ok = fila(CONTROL, SIN_LAGS)
ruido = fila(RUIDO, BASE_CURADA)
vale = historia_ok['d_M'] > 0 and historia_ok['ic95_inf_M'] > 0
sin_premio = ruido['d_M'] < EFECTO_BUSCADO and ruido['ic95_inf_M'] <= 0
sirve = bool(vale and sin_premio)
print('1. ¿El banco sirve?', 'Sí' if sirve else 'No: las candidatas no se leen')
print(f"   AC - ACS: {historia_ok['d_M']:+.2f} M ({historia_ok['ic95_inf_M']:.2f} a {historia_ok['ic95_sup_M']:.2f})")
print(f"   ACK - ACH: {ruido['d_M']:+.2f} M ({ruido['ic95_inf_M']:.2f} a {ruido['ic95_sup_M']:.2f})")
error = comparacion[comparacion['config'].isin(CANDIDATAS)]['error_M'].median()
print(f'   Error del promedio mensual en las candidatas: {error:.2f} M (la cuenta aproximada daba 2,4 M)')
print()

umbral = max(0.0, ruido['d_M'])
suman = []
print(f'2. ¿Qué candidatas suman? Tienen que dar más de {umbral:+.2f} M contra ACH.')
for c in sorted(CANDIDATAS, key=lambda c: -fila(c, BASE_CURADA)['d_M']):
    r = fila(c, BASE_CURADA)
    pasa = sirve and r['d_M'] > umbral
    suman += [c] if pasa else []
    print(f"   {c}: {r['d_M']:+.2f} M ({r['ic95_inf_M']:.2f} a {r['ic95_sup_M']:.2f}) · 201905-08: {r['d4_M']:+.2f} M · {'suma' if pasa else 'no suma' if sirve else 'no se lee'}")
print()
print('Candidatas para acumular:', ', '.join(suman) if suman else 'ninguna')

iteraciones = pd.DataFrame([{
    'config': c, 'hipotesis': HIPOTESIS[c], 'columnas_nuevas': B[c][0].shape[1] if c in B else 0,
    'ganancia_banda_M': round(media_meses(c, MESES_BANCO), 3),
    'd_contra_ACH_M': round(fila(c, BASE_CURADA)['d_M'], 3) if c in LISTA else np.nan,
    'resultado': ('suma' if c in suman else 'no suma' if sirve else 'no se lee') if c in CANDIDATAS else 'calibración',
} for c in CONFIGS])
iteraciones.to_csv(RESULTADOS / '13_iteraciones.tsv', sep='\t', index=False)